In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

import os

figure_dir = "figures/revision/figure-2"
setup_plotting(figure_dir, display_dpi=300, save_dpi=300)

# import squarify

import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import seaborn as sns
from matplotlib.patches import Patch

In [ ]:
data_dir = "data/xenium/processed"
path = f"{data_dir}/06.2-kidney_tcr_filtered.h5ad"
adata = sc.read_h5ad(path)
adata.obs["condition"].value_counts()

In [ ]:
cond_dic = {
    "Control": [
        "XETG00088__0029040__Region_2__20240719__095641",
        "XETG00088__0029040__Region_3__20240719__095641",
        "XETG00088__0029041__Region_1__20240719__095642",
    ],
    "ANCA": [
        "XETG00088__0029040__Region_4__20240719__095642",
        "XETG00088__0029040__Region_5__20240719__095642",
        "XETG00088__0029040__Region_6__20240719__095642",
        "XETG00088__0029040__Region_7__20240719__095642",
        "XETG00088__0029041__Region_3__20240719__095642",
        "XETG00088__0029041__Region_4__20240719__095642",
        "XETG00088__0029041__Region_5__20240719__095642",
        "XETG00088__0029041__Region_6__20240719__095642",
        "XETG00088__0029041__Region_7__20240719__095642",
        "XETG00088__0029041__Region_8__20240719__095642",
    ],
    "BK-Virus": ["XETG00088__0029041__Region_2__20240719__095642"],
}
inverted_dict = {
    sample_id: condition
    for condition, sample_ids in cond_dic.items()
    for sample_id in sample_ids
}

In [ ]:
ANCA_mapping = {sample: f"ANCA {i}" for i, sample in enumerate(cond_dic["ANCA"])}
ANCA_mapping

In [ ]:
tmp = adata[adata.obs["cell_type_l1"] == "T"]

In [ ]:
# only concentrate on the vgenes
av = [x for x in tmp.var_names if x.startswith("TRAV")]
bv = [x for x in tmp.var_names if x.startswith("TRBV")]
gv = [x for x in tmp.var_names if x.startswith("TRGV")]
dv = [x for x in tmp.var_names if x.startswith("TRDV")]

# per sample Analysis AVBV

In [ ]:
def get_dominant_v(tmp, vgenes):
    array = tmp[:, vgenes].X.toarray()
    max_indices = np.argmax(array, axis=1)
    # max_values = array[np.arange(array.shape[0]), max_indices]
    # pdb.set_trace()
    sorted_rows = np.sort(array, axis=1)[:, -2:]
    ties = sorted_rows[:, 0] == sorted_rows[:, 1]
    result = np.where(ties, -1, max_indices)
    # pdb.set_trace()
    return [vgenes[i] if i != -1 else "None" for i in result]

In [ ]:
tmp.obs["AV"] = get_dominant_v(tmp, av)
tmp.obs["BV"] = get_dominant_v(tmp, bv)

# per sample Analysis AVBV - Expanded Tcells

In [ ]:
def create_expansionstack(smp, unique=False):
    smp = smp[(smp.obs[["AV", "BV"]] != "None").all(1)]
    avbv_vc = smp.obs[["AV", "BV"]].value_counts()
    exp_dic = [avbv_vc[avbv_vc == 1].sum(), avbv_vc[avbv_vc >= 2].sum()]
    if unique:
        exp_dic = [avbv_vc[avbv_vc == 1].shape[0], avbv_vc[avbv_vc >= 2].shape[0]]
    return exp_dic[::]

# per sample Analysis AVBV - Expanded cells + CD4vsCD8

In [ ]:
def get_anca_control_bar(tmp, cond_dic, unique=False):
    anca_bar = []
    for sample in cond_dic["ANCA"]:
        smp = tmp[tmp.obs["sample"] == sample]
        smp4 = smp[smp.obs.tcell_subtype == "CD4+"]
        smp8 = smp[smp.obs.tcell_subtype == "CD8+"]
        x4 = create_expansionstack(smp4, unique)
        x8 = create_expansionstack(smp8, unique)
        anca_bar.append([x4, x8])
    anca_bar = np.array(anca_bar)
    control_bar = []
    for sample in cond_dic["Control"]:
        smp = tmp[tmp.obs["sample"] == sample]
        smp4 = smp[smp.obs.tcell_subtype == "CD4+"]
        smp8 = smp[smp.obs.tcell_subtype == "CD8+"]
        x4 = create_expansionstack(smp4, unique)
        x8 = create_expansionstack(smp8, unique)
        control_bar.append([x4, x8])
    control_bar = np.array(control_bar)
    return anca_bar, control_bar

In [ ]:
anca_bar, control_bar = get_anca_control_bar(tmp, cond_dic)

In [ ]:
anca_bar.shape

In [ ]:
from spatial_tcr.colors import colors_sub

cd4color = colors_sub["CD4+"]
cd8color = colors_sub["CD8+"]

In [ ]:
scaler = 0.925
fig, ax = plt.subplots(figsize=(4.5 * scaler, 3 * scaler))

x = list(range(anca_bar.shape[0]))
x_shifted_left = [3.8 + _ for _ in x]
x_shifted_right = [4.1 + _ for _ in x]

control_x_left = [-0.15, 0.85, 1.85]
control_x_right = [0.15, 1.15, 2.15]

# expanded_color = 'darkred'
# singlet_edgecolor = 'purple'
bar_width = 0.3

# Plotting
bottom = np.zeros(anca_bar.shape[0])
bottomc = np.zeros(3)

for i in range(2):  # Singlets and Expanded
    # Singlets: only draw border
    if i == 0:
        ax.bar(
            x_shifted_left,
            anca_bar[:, 0, i],
            fill=False,
            edgecolor=cd4color,
            bottom=bottom,
            width=bar_width,
            label="Singlets",
        )
        ax.bar(
            control_x_left,
            control_bar[:, 0, i],
            fill=False,
            edgecolor=cd4color,
            bottom=bottomc,
            width=bar_width,
        )
    # Expanded: filled
    else:
        ax.bar(
            x_shifted_left,
            anca_bar[:, 0, i],
            color=cd4color,
            edgecolor=cd4color,
            bottom=bottom,
            width=bar_width,
            label="Expanded",
        )
        ax.bar(
            control_x_left,
            control_bar[:, 0, i],
            color=cd4color,
            edgecolor=cd4color,
            bottom=bottomc,
            width=bar_width,
        )

    bottom += anca_bar[:, 0, i]
    bottomc += control_bar[:, 0, i]

bottom = np.zeros(anca_bar.shape[0])
bottomc = np.zeros(3)

for i in range(2):  # CD8+
    if i == 0:
        ax.bar(
            x_shifted_right,
            anca_bar[:, 1, i],
            fill=False,
            edgecolor=cd8color,
            bottom=bottom,
            width=bar_width,
        )
        ax.bar(
            control_x_right,
            control_bar[:, 1, i],
            fill=False,
            edgecolor=cd8color,
            bottom=bottomc,
            width=bar_width,
        )
    else:
        ax.bar(
            x_shifted_right,
            anca_bar[:, 1, i],
            color=cd8color,
            edgecolor=cd8color,
            bottom=bottom,
            width=bar_width,
        )
        ax.bar(
            control_x_right,
            control_bar[:, 1, i],
            color=cd8color,
            edgecolor=cd8color,
            bottom=bottomc,
            width=bar_width,
        )

    bottom += anca_bar[:, 1, i]
    bottomc += control_bar[:, 1, i]

# Labels and legend
# ax.set_xlabel("samples")
ax.set_ylabel("number of T cells with TRAV+TRBV", fontsize=8)
# ax.set_title("T cells with complete TCR")
ax.set_xticks(list(range(3)) + [x + 4 for x in range(10)])
ax.set_xticklabels(
    [f"Control {x}" for x in range(3)] + [f"ANCA {x}" for x in range(10)], rotation=90
)
ax.tick_params(axis="both", labelsize=6)

# Legends
color_patches = [
    Patch(facecolor="none", edgecolor="black", label="=1"),
    Patch(facecolor="k", label=">1"),
]
group_patches = [
    Patch(facecolor=cd4color, label="CD4+"),  # left
    Patch(facecolor=cd8color, label="CD8+"),  # right
]
legend1 = ax.legend(
    handles=color_patches,
    title="clonal size",
    loc="upper left",
    bbox_to_anchor=(0.0, 1),
    frameon=False,
    fontsize=6,
    title_fontsize=6,
)
legend2 = ax.legend(
    handles=group_patches,
    title="groups",
    loc="upper left",
    bbox_to_anchor=(0.0, 0.8),
    frameon=False,
    fontsize=6,
    title_fontsize=6,
)
ax.add_artist(legend1)
sns.despine(ax=ax)
# plt.tight_layout()
# plt.show()
plt.savefig(
    os.path.join(figure_dir, "Nr_completeTCR_persample_expanded.pdf"),
    dpi=300,
    bbox_inches="tight",
)

# Clonesize and TRAV+TRBV combinations

In [ ]:
from collections import Counter

# Initialize counters for CD4+ and CD8+ clone sizes
agg4 = Counter()
agg8 = Counter()

for sample in cond_dic["ANCA"]:
    smp = tmp[(tmp.obs[["AV", "BV"]] != "None").all(1)].copy()
    smp = smp[smp.obs["sample"] == sample].copy()

    # CD4+ T cells
    smp4 = smp[smp.obs.tcell_subtype == "CD4+"]
    counts4 = smp4.obs[["AV", "BV"]].value_counts()
    agg4 += Counter(counts4.values)

    # CD8+ T cells
    smp8 = smp[smp.obs.tcell_subtype == "CD8+"]
    counts8 = smp8.obs[["AV", "BV"]].value_counts()
    agg8 += Counter(counts8.values)

# Convert to sorted lists for plotting
smp4x, smp4y = zip(*sorted(agg4.items()))
smp8x, smp8y = zip(*sorted(agg8.items()))

In [ ]:
# Plotting
# Custom softened colors
# Set where to break the y-axis
ymax_top = max(max(smp4y), max(smp8y))
break_point = 50
break_height = 1  # height of the break zone
scaler = 0.925
fig, (ax1, ax2) = plt.subplots(
    2,
    1,
    sharex=True,
    figsize=(4.5 * scaler, 3 * scaler),
    gridspec_kw={"height_ratios": [1, 3]},
)

# Top plot (high values only)
ax1.bar([x - 0.2 for x in smp4x], smp4y, color=cd4color, width=0.4)
ax1.bar([x + 0.2 for x in smp8x], smp8y, color=cd8color, width=0.4, edgecolor="k")
ax1.set_ylim(break_point + break_height, ymax_top + 50)  # upper part of the y-axis

# Bottom plot (normal range)
ax2.bar(
    [x - 0.2 for x in smp4x], smp4y, color=cd4color, label="CD4+ T cells", width=0.4
)
ax2.bar(
    [x + 0.2 for x in smp8x],
    smp8y,
    color=cd8color,
    label="CD8+ T cells",
    edgecolor="k",
    width=0.4,
)
ax2.set_ylim(0, break_point)  # lower part of the y-axis

# Hide spines between plots
ax1.spines["bottom"].set_visible(False)
ax2.spines["top"].set_visible(False)
ax1.tick_params(labeltop=False, labelsize=6)  # Don't show ticks on top
ax2.xaxis.tick_bottom()

# Add break marks
d = 0.5  # size of diagonal lines
kwargs = {
    "marker": [(-1, -d), (1, d)],
    "markersize": 8,
    "linestyle": "none",
    "color": "k",
    "mec": "k",
    "mew": 1,
    "clip_on": False,
}
# left edge only
ax1.plot([0], [0], transform=ax1.transAxes, **kwargs)
ax2.plot([0], [1], transform=ax2.transAxes, **kwargs)

# Labels and legend
ax2.set_xlabel("clone size", fontsize=8)
# ax1.set_ylabel('# Unique TRAV+TRBV')
ax2.set_ylabel("number of unique TRAV+TRBV", fontsize=8)
# ax1.set_title("Clone size distribution aggregated over all ANCA samples")
ax2.legend(fontsize=6, title_fontsize=6, frameon=False)
ax2.tick_params(axis="both", labelsize=6)

# plt.tight_layout()
sns.despine(ax=ax1, bottom=True, right=True, top=True)

# remove ax1 xticks
ax1.tick_params(axis="x", length=0)
ax1.tick_params(axis="y", labelsize=6)

sns.despine(ax=ax2)
plt.subplots_adjust(hspace=0.05)
plt.savefig(
    os.path.join(figure_dir, "02_allANCA_clonesize_brokenY.pdf"),
    dpi=300,
    bbox_inches="tight",
)
# plt.show()